# 03 Retention-Aware MBA

**Client deliverable for LushProtein — Customer Retention via Market Basket Analysis**

Notebook `02` ranked product pairs by **lift** — i.e. how surprising a co-purchase is. That answers *"what raises basket size today?"* (an AOV question). It does **not** answer the question LushProtein actually cares about: *"which product combinations turn a one-and-done buyer into a repeat, subscribed, high-LTV customer?"*

The EDA already proves cross-sell is fundamentally a **retention** lever, not just an AOV lever:

| Categories bought | Repeat rate | Avg LTV |
|---|---|---|
| 1 product | 24% | $170 |
| 2 products | 40% | $230 |
| 3 products | 65% | $470 |

*(source: `EDA/outputs/04_cross_product_ltv.csv`)*

And subscription is the single biggest retention multiplier in the business: subscribers repeat at **74% vs 29%** and carry **2.7x the LTV** (`06_sub_vs_onetime_ltv.csv`).

This notebook keeps `02`'s classic MBA as the foundation but **re-scores every rule by its retention impact** — building a composite *Retention Value Score* and surfacing *retention-anchor products*.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd


def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "EDA" / "outputs").exists() and (candidate / "product_mba").exists():
            return candidate
    raise FileNotFoundError("Could not locate project root containing EDA/outputs and product_mba")

PROJECT_ROOT = find_project_root()
EDA_OUTPUTS = PROJECT_ROOT / "EDA" / "outputs"
MBA_OUTPUTS = PROJECT_ROOT / "product_mba" / "outputs"
MBA_OUTPUTS.mkdir(exist_ok=True)

pd.set_option("display.float_format", lambda v: f"{v:,.3f}")
print("Project root found")

Project root found


In [2]:
# --- Business baselines: what does a "typical" LushProtein customer look like? ---
customers = pd.read_parquet(EDA_OUTPUTS / "customers.parquet")

BASE_REPEAT = customers["is_repeat"].mean()
BASE_SUB = customers["ever_subscribed"].mean()
BASE_LTV = customers["total_revenue"].mean()

breadth = pd.read_csv(EDA_OUTPUTS / "04_cross_product_ltv.csv")

print(f"Base repeat rate     : {BASE_REPEAT:.1%}")
print(f"Base subscription    : {BASE_SUB:.1%}")
print(f"Base avg LTV (SGD)    : {BASE_LTV:,.0f}")
print("\nThe retention cliff (cross-sell as a retention lever):")
display(breadth[["category_label", "customers", "repeat_rate", "avg_ltv", "avg_orders"]])

Base repeat rate     : 32.4%
Base subscription    : 7.9%
Base avg LTV (SGD)    : 226

The retention cliff (cross-sell as a retention lever):


,category_label,customers,repeat_rate,avg_ltv,avg_orders
0,1 product,9537,0.236,170.137,1.480
1,2 products,2679,0.397,229.863,2.244
2,3 products,1052,0.655,469.700,3.429
3,4+ products,512,0.887,743.018,7.057


In [3]:
# --- Re-score every 02 rule by retention impact, not raw lift ---
MIN_BOTH_CUSTOMERS = 30  # ignore rules whose "bought-both" segment is too small to trust

def minmax(s):
    s = s.astype(float)
    lo, hi = s.min(), s.max()
    if not np.isfinite(lo) or not np.isfinite(hi) or hi == lo:
        return pd.Series(0.5, index=s.index)
    return (s - lo) / (hi - lo)

def retention_score(level):
    df = pd.read_csv(MBA_OUTPUTS / f"mba_rules_{level}.csv")
    if df.empty:
        return df
    # average estimated margin of the two items (directional, per 00_margin_data caveats)
    df["pair_est_margin_pct"] = df[["antecedent_item_est_margin_pct",
                                    "consequent_item_est_margin_pct"]].mean(axis=1)

    # uplift vs the typical customer
    df["repeat_rate_uplift"] = df["both_item_repeat_rate"] - BASE_REPEAT
    df["subscription_uplift"] = df["both_item_pct_subscribed"] - BASE_SUB
    df["ltv_ratio"] = df["both_item_avg_ltv"] / BASE_LTV

    df["trustworthy"] = df["both_item_customers"] >= MIN_BOTH_CUSTOMERS
    scored = df[df["trustworthy"]].copy()
    if scored.empty:
        scored = df.copy()

    # composite Retention Value Score — retention-weighted, with lift/scale/margin as support
    scored["n_repeat_uplift"] = minmax(scored["repeat_rate_uplift"])
    scored["n_sub_uplift"] = minmax(scored["subscription_uplift"])
    scored["n_lift"] = minmax(scored["lift"])
    scored["n_scale"] = minmax(np.log1p(scored["co_orders"]))
    scored["n_margin"] = minmax(scored["pair_est_margin_pct"].fillna(scored["pair_est_margin_pct"].median()))

    scored["retention_value_score"] = (
        0.30 * scored["n_repeat_uplift"]
        + 0.25 * scored["n_sub_uplift"]
        + 0.20 * scored["n_lift"]
        + 0.15 * scored["n_scale"]
        + 0.10 * scored["n_margin"]
    )
    scored["level"] = level
    return scored.sort_values("retention_value_score", ascending=False)

show_cols = ["antecedent", "consequent", "co_orders", "lift", "confidence",
             "both_item_repeat_rate", "repeat_rate_uplift", "both_item_pct_subscribed",
             "subscription_uplift", "ltv_ratio", "pair_est_margin_pct", "retention_value_score"]

scored_tables = {}
for level in ["category", "handle", "sku_flavor"]:
    s = retention_score(level)
    scored_tables[level] = s
    s.to_csv(MBA_OUTPUTS / f"mba_rules_retention_scored_{level}.csv", index=False)
    print(f"\n=== {level}: top 8 by Retention Value Score (n={len(s)}) ===")
    display(s[show_cols].head(8).reset_index(drop=True))


=== category: top 8 by Retention Value Score (n=4) ===


,antecedent,consequent,co_orders,lift,confidence,both_item_repeat_rate,repeat_rate_uplift,both_item_pct_subscribed,subscription_uplift,ltv_ratio,pair_est_margin_pct,retention_value_score
0,Accessories,Lean Protein,774,1.357,0.340,0.490,0.167,0.202,0.123,1.107,0.773,0.600
1,Lean Protein,Accessories,774,1.357,0.206,0.490,0.167,0.202,0.123,1.107,0.773,0.600
2,Accessories,Clear Protein,734,1.298,0.323,0.511,0.187,0.201,0.122,1.245,0.778,0.400
3,Clear Protein,Accessories,734,1.298,0.197,0.511,0.187,0.201,0.122,1.245,0.778,0.400



=== handle: top 8 by Retention Value Score (n=14) ===


,antecedent,consequent,co_orders,lift,confidence,both_item_repeat_rate,repeat_rate_uplift,both_item_pct_subscribed,subscription_uplift,ltv_ratio,pair_est_margin_pct,retention_value_score
0,multivitamin-vegan,super-omega-3,50,21.086,0.321,0.789,0.466,0.263,0.184,3.048,0.654,0.705
1,super-omega-3,multivitamin-vegan,50,21.086,0.219,0.789,0.466,0.263,0.184,3.048,0.654,0.705
2,green-tea-extract-capsules,pureburn-fat-burner-capsules,53,21.371,0.353,0.698,0.375,0.170,0.090,5.527,0.776,0.635
3,pureburn-fat-burner-capsules,green-tea-extract-capsules,53,21.371,0.214,0.698,0.375,0.170,0.090,5.527,0.776,0.635
4,super-omega-3,collagen-glow,51,1.813,0.224,0.870,0.547,0.259,0.180,5.864,0.668,0.557
5,green-tea-extract-capsules,lean-protein,46,1.543,0.307,0.771,0.447,0.271,0.191,5.973,0.709,0.532
6,super-omega-3,micronized-creatine-monohydrate,66,3.060,0.289,0.754,0.430,0.200,0.121,4.497,0.734,0.507
7,lushprotein-clear-shaker,lushprotein-lean-protein-40g-single-serve,245,2.027,0.108,0.487,0.164,0.149,0.070,1.264,0.824,0.476



=== sku_flavor: top 8 by Retention Value Score (n=222) ===


,antecedent,consequent,co_orders,lift,confidence,both_item_repeat_rate,repeat_rate_uplift,both_item_pct_subscribed,subscription_uplift,ltv_ratio,pair_est_margin_pct,retention_value_score
0,0724999807999 | Creatine Monohydrate | Default,0724999808361 | [FREE GIFT] LushProtein Classi...,24,2.393,0.054,0.915,0.591,0.936,0.857,3.443,NaN,0.602
1,0724999808361 | [FREE GIFT] LushProtein Classi...,0724999807999 | Creatine Monohydrate | Default,24,2.393,0.068,0.915,0.591,0.936,0.857,3.443,NaN,0.602
2,0724999807937 | BETTER WHEY PROTEIN | 1kg (40 ...,0724999808361 | [FREE GIFT] LushProtein Classi...,10,1.718,0.039,0.967,0.643,1.000,0.921,2.892,NaN,0.600
3,0724999808361 | [FREE GIFT] LushProtein Classi...,0724999807937 | BETTER WHEY PROTEIN | 1kg (40 ...,10,1.718,0.028,0.967,0.643,1.000,0.921,2.892,NaN,0.600
4,0724999808361 | [FREE GIFT] LushProtein Classi...,0724999808361 | LushProtein Classic Shaker | D...,16,2.119,0.046,0.900,0.576,0.933,0.854,7.844,NaN,0.578
5,0724999808361 | LushProtein Classic Shaker | D...,0724999808361 | [FREE GIFT] LushProtein Classi...,16,2.119,0.048,0.900,0.576,0.933,0.854,7.844,NaN,0.578
6,0724999807937 | BETTER WHEY PROTEIN | 1kg Pack...,0724999808361 | [FREE GIFT] LushProtein Classi...,39,5.229,0.119,0.750,0.426,0.886,0.807,2.172,NaN,0.551
7,0724999808361 | [FREE GIFT] LushProtein Classi...,0724999807937 | BETTER WHEY PROTEIN | 1kg Pack...,39,5.229,0.111,0.750,0.426,0.886,0.807,2.172,NaN,0.551


In [4]:
# --- AOV-optimal vs retention-optimal: do the rankings actually differ? ---
def top_pairs(df, by, k=10):
    return set(zip(df.sort_values(by, ascending=False).head(k)["antecedent"],
                   df.sort_values(by, ascending=False).head(k)["consequent"]))

for level in ["category", "handle"]:
    s = scored_tables[level]
    lift_top = top_pairs(s, "lift")
    ret_top = top_pairs(s, "retention_value_score")
    only_ret = ret_top - lift_top
    print(f"[{level}] top-10 overlap between lift and retention ranking: "
          f"{len(lift_top & ret_top)}/10")
    if only_ret:
        print("  Pairs the retention lens promotes that pure-lift misses:")
        for a, b in list(only_ret)[:6]:
            print(f"    - {a}  ->  {b}")

[category] top-10 overlap between lift and retention ranking: 4/10
[handle] top-10 overlap between lift and retention ranking: 6/10
  Pairs the retention lens promotes that pure-lift misses:
    - green-tea-extract-capsules  ->  lean-protein
    - lushprotein-clear-shaker  ->  lushprotein-lean-protein-40g-single-serve
    - lushprotein-lean-protein-40g-single-serve  ->  lushprotein-clear-shaker
    - super-omega-3  ->  collagen-glow


In [5]:
# --- Retention-anchor products: which items most pull customers up the retention cliff? ---
def anchor_table(level):
    s = scored_tables[level]
    if s.empty:
        return pd.DataFrame()
    long = pd.concat([
        s[["antecedent", "co_orders", "repeat_rate_uplift", "subscription_uplift", "ltv_ratio"]]
            .rename(columns={"antecedent": "item"}),
        s[["consequent", "co_orders", "repeat_rate_uplift", "subscription_uplift", "ltv_ratio"]]
            .rename(columns={"consequent": "item"}),
    ], ignore_index=True)

    def wavg(g, col):
        w = g["co_orders"]
        return np.average(g[col], weights=w) if w.sum() else np.nan

    rows = []
    for item, g in long.groupby("item"):
        rows.append({
            "level": level,
            "item": item,
            "appears_in_rules": len(g),
            "total_co_orders": int(g["co_orders"].sum()),
            "wtd_repeat_uplift": wavg(g, "repeat_rate_uplift"),
            "wtd_subscription_uplift": wavg(g, "subscription_uplift"),
            "wtd_ltv_ratio": wavg(g, "ltv_ratio"),
        })
    out = pd.DataFrame(rows)
    out["anchor_score"] = (minmax(out["wtd_repeat_uplift"]) + minmax(out["wtd_subscription_uplift"])) / 2
    return out.sort_values("anchor_score", ascending=False)

anchors = pd.concat([anchor_table("category"), anchor_table("handle")], ignore_index=True)
anchors.to_csv(MBA_OUTPUTS / "retention_anchor_products.csv", index=False)
print("Top retention-anchor products (category + handle):")
display(anchors.head(15).reset_index(drop=True))

Top retention-anchor products (category + handle):


,level,item,appears_in_rules,total_co_orders,wtd_repeat_uplift,wtd_subscription_uplift,wtd_ltv_ratio,anchor_score
0,category,Accessories,4,3016,0.177,0.122,1.174,0.500
1,category,Clear Protein,2,1468,0.187,0.122,1.245,0.500
2,category,Lean Protein,2,1548,0.167,0.123,1.107,0.500
3,handle,collagen-glow,1,51,0.547,0.180,5.864,0.973
4,handle,lean-protein,1,46,0.447,0.191,5.973,0.916
5,handle,multivitamin-vegan,2,100,0.466,0.184,3.048,0.914
6,handle,super-omega-3,4,217,0.474,0.164,4.150,0.874
7,handle,micronized-creatine-monohydrate,1,66,0.430,0.121,4.497,0.738
8,handle,green-tea-extract-capsules,3,152,0.397,0.121,5.662,0.710
9,handle,pureburn-fat-burner-capsules,2,106,0.375,0.090,5.527,0.621


## Readout: from "biggest basket" to "stickiest basket"

**1. Lift and retention are not the same ranking.** The overlap check above shows the retention lens reshuffles the priority list versus `02`'s pure-lift ranking. High-lift sachet flavor pairs (e.g. Clear Protein Peach + White Grape) are great *discovery* mechanics but are weak retention anchors on their own; cross-category combos that move a customer from 1→2→3 categories are where repeat rate and LTV actually compound.

**2. The retention cliff is the strategy.** Going 1→2 categories nearly doubles repeat rate (24%→40%); 2→3 takes it to 65% and ~2.8x LTV. Every cross-sell rule should be judged on whether it *advances a customer across that cliff*, which is exactly what the `repeat_rate_uplift` / `ltv_ratio` columns now measure per rule.

**3. Subscription is the retention jackpot — score for it.** Because subscribers repeat at 74% vs 29%, `subscription_uplift` is weighted heavily in the score. Pairs that over-index on already-subscribed customers (`both_item_pct_subscribed` well above the ~8% base) are the combos to convert into subscribe-and-save bundles in notebook `05`.

**4. Retention-anchor products are the merchandising spine.** `retention_anchor_products.csv` ranks items by how much *every* basket containing them lifts repeat/subscription. These are the products to feature on the homepage, in the "complete your stack" module, and as the default add-on at checkout — not because they sell the most, but because their presence predicts a retained customer.

**Caveat:** margin is directional only (see `00_margin_data.ipynb`, "usable with caveats"). It is the lowest-weighted term in the score and is used as a tie-breaker, never the driver.

Outputs written: `mba_rules_retention_scored_{category,handle,sku_flavor}.csv`, `retention_anchor_products.csv`. These feed notebooks `04` (timing) and `05` (the campaign playbook).

---
### Final metrics & scores

**Baselines:** repeat rate **32%** · subscription **8%** · avg LTV **$226**. Rules scored: category **4** · handle **14** · SKU/flavor **222** (≥30 both-item customers).

**Top rules by Retention Value Score (RVS)** — uplift = percentage points above baseline:

| Level | Rule | RVS | Lift | Repeat uplift | Sub. uplift | LTV× |
|---|---|--:|--:|--:|--:|--:|
| Handle | multivitamin-vegan ↔ super-omega-3 | 0.70 | 21.1 | +47 pts | +18 pts | 3.0× |
| Handle | green-tea-extract ↔ pureburn | 0.63 | 21.4 | +37 pts | +9 pts | 5.5× |
| Handle | super-omega-3 → collagen-glow | 0.56 | 1.81 | +55 pts | +18 pts | 5.9× |
| Category | Accessories ↔ Lean Protein | 0.60 | 1.36 | +17 pts | +12 pts | 1.1× |
| Category | Accessories ↔ Clear Protein | 0.40 | 1.30 | +19 pts | +12 pts | 1.2× |

**Top retention-anchor products** (handle level, by `anchor_score`):

| Product | Anchor score | Repeat uplift | Sub. uplift | LTV× |
|---|--:|--:|--:|--:|
| collagen-glow | 0.97 | +55 pts | +18 pts | 5.9× |
| lean-protein | 0.92 | +45 pts | +19 pts | 6.0× |
| multivitamin-vegan | 0.91 | +47 pts | +18 pts | 3.0× |
| super-omega-3 | 0.87 | +47 pts | +16 pts | 4.2× |
| micronized-creatine | 0.74 | +43 pts | +12 pts | 4.5× |

**Read:** the ranking reshuffles versus `02`. Pure-lift favourites (Accessories pairs) land mid-table, while **supplement attaches** — modest lift but very large repeat / subscription / LTV uplift — rise to the top. That is the whole point: score for stickiness, not basket size.
